In [ ]:
from pathlib import Path
import os
import pandas as pd

from FIT_python.config import PROJECT_ROOT, RAW_DIR, CLEANED_DIR
from FIT_python.data_split_and_summary.dataset_summary import generate_dataset_overview

os.chdir(PROJECT_ROOT)
from FIT_python.Visualisations.plots_utils import (
    plot_feature_correlation_matrix,
    plot_sex_boxplots,
    plot_individual_boxplots,
    plot_umap_scatter,
    select_top_features,
)
from FIT_python.general_pipeline_steps.dimensionality_reduction_wrapper import DimensionalityReducerTransformer
from sklearn.model_selection import StratifiedKFold

EXP_DIR = Path('experiments/fit_start_to_finish')
EXP_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
overview_df = generate_dataset_overview(RAW_DIR, EXP_DIR/'dataset_overview.csv')
overview_df

In [ ]:
from FIT_python.data_split_and_summary.datasplit_and_summary_wraper import SplitWrapper
from FIT_python.data_split_and_summary.summary_data_wrapper import run_summary
from FIT_python.config import SPLITS_DIR

SplitWrapper().split_all()
run_summary(SPLITS_DIR, EXP_DIR/'split_summary.csv', EXP_DIR/'split_fig')


In [ ]:
from IPython.display import Image, display
display(Image(filename=EXP_DIR/'split_fig/summary_all.png'))
display(Image(filename=EXP_DIR/'split_fig/split_proportions.png'))
display(Image(filename=EXP_DIR/'split_fig/eurasian_otter_summary.png'))


In [ ]:
otter_df = pd.read_parquet(CLEANED_DIR / 'Eurasian_Otter.parquet')
fig_path = plot_feature_correlation_matrix(otter_df, EXP_DIR)
fig_path

In [ ]:
top_sex = select_top_features(otter_df, 'sex', k=4)
plot_sex_boxplots(otter_df, top_sex, EXP_DIR, 'sex_boxplots.png')

In [ ]:
top_ind = select_top_features(otter_df, 'individual_id', k=4)
plot_individual_boxplots(otter_df, top_ind, EXP_DIR, 'individual_boxplots.png')

In [ ]:
features = [c for c in otter_df.columns if c.startswith(('dist','ang','t'))]
X = otter_df[features].astype(float)
y_sex = otter_df['sex']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
emb_list = []
for train_idx, test_idx in cv.split(X, y_sex):
    dr = DimensionalityReducerTransformer(method='umap', supervised=True)
    dr.fit(X.iloc[train_idx], y_sex.iloc[train_idx])
    emb = dr.transform(X.iloc[test_idx])
    emb_list.append(pd.DataFrame(emb, index=otter_df.index[test_idx], columns=dr.get_feature_names_out()))
emb_sex = pd.concat(emb_list).loc[otter_df.index]
plot_umap_scatter(otter_df.assign(sex_mapped=otter_df['sex']), emb_sex, EXP_DIR, 'umap_sex.png')

In [ ]:
y_ind = otter_df['individual_id']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
emb_list = []
for train_idx, test_idx in cv.split(X, y_ind):
    dr = DimensionalityReducerTransformer(method='umap', supervised=True)
    dr.fit(X.iloc[train_idx], y_ind.iloc[train_idx])
    emb = dr.transform(X.iloc[test_idx])
    emb_list.append(pd.DataFrame(emb, index=otter_df.index[test_idx], columns=dr.get_feature_names_out()))
emb_ind = pd.concat(emb_list).loc[otter_df.index]
plot_umap_scatter(otter_df.assign(sex_mapped=otter_df['sex']), emb_ind, EXP_DIR, 'umap_individual.png')